In [80]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import calibration_curve
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, roc_auc_score

In [81]:
DATA_START_SEASON = 2009
MODEL_START_SEASON = 2010
VAL_SEASON = 2024
TEST_SEASON = 2025
SEASONS = list(range(DATA_START_SEASON, TEST_SEASON + 1))


In [82]:
pbp_columns = [
    'game_id',
    'posteam',
    'defteam',
    'epa',
    'success',
    'pass',
    'rush',
    'qb_kneel',
    'qb_spike',
    'two_point_attempt',
    'interception',
    'fumble_lost',
    'sack',
    'qb_dropback',
    'pass_attempt',
    'qb_hit',
    'passer_player_id',
    'passer_player_name',
]

schedule_columns = [
    'game_id',
    'season',
    'week',
    'gameday',
    'game_type',
    'home_team',
    'away_team',
    'home_score',
    'away_score',
    'home_rest',
    'away_rest',
    'location',
    'home_qb_id',
    'away_qb_id',
    'home_qb_name',
    'away_qb_name',
]

In [83]:
pbp = nfl.load_pbp(SEASONS).select(pbp_columns).to_pandas()
schedule = nfl.load_schedules(SEASONS).select(schedule_columns).to_pandas()

In [84]:
TEAM_ABBR_MAP = {
    "SD": "LAC",
    "OAK": "LV",
    "STL": "LA",
}
pbp[["posteam", "defteam"]] = (
    pbp[["posteam", "defteam"]]
    .replace(TEAM_ABBR_MAP)
)
schedule[["home_team", "away_team"]] = (
    schedule[["home_team", "away_team"]]
    .replace(TEAM_ABBR_MAP)
)

In [85]:
scrimmage_plays = pbp.loc[
    pbp['posteam'].notna()
    & pbp['defteam'].notna()
    & pbp['epa'].notna()
    & pbp['success'].notna()
    & (pbp['pass'].eq(1) | pbp['rush'].eq(1))
    & pbp['qb_kneel'].fillna(0).ne(1)
    & pbp['qb_spike'].fillna(0).ne(1)
    & pbp['two_point_attempt'].fillna(0).ne(1)
].copy()

In [86]:
offense = (
    scrimmage_plays
    .groupby(['game_id', 'posteam'])
    .agg(
        off_epa = ('epa', 'mean'),
        off_success_rate = ('success', 'mean')
    )
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [87]:
passing = (
    scrimmage_plays.loc[scrimmage_plays["pass"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(pass_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)
rush = (
    scrimmage_plays.loc[scrimmage_plays["rush"].eq(1)]
    .groupby(['game_id', 'posteam'])
    .agg(rush_epa = ('epa', 'mean'))
    .reset_index()
    .rename(columns={'posteam': 'team'})
)

In [88]:
defense = (
    scrimmage_plays
    .groupby(['game_id', 'defteam'])
    .agg(
        def_epa_allowed = ('epa', 'mean'),
        def_success_rate_allowed = ('success', 'mean'),
    )
    .reset_index()
    .rename(columns={'defteam': 'team'})
)

In [89]:
home_points = (
    schedule[['game_id', 'home_team', 'home_score', 'away_score']]
    .rename(columns={'home_team': 'team', 'home_score': 'points_scored', 'away_score': 'points_allowed'})
)
away_points = (
    schedule[['game_id', 'away_team', 'home_score', 'away_score']]
    .rename(columns={'away_team': 'team', 'home_score': 'points_allowed', 'away_score': 'points_scored'})
)
points = pd.concat([home_points, away_points], ignore_index=True)
points.sort_values(by=['game_id', 'team'], inplace=True)
points.reset_index(drop=True, inplace=True)


In [99]:
points['win'] = np.select(
    [
        points['points_scored'] > points['points_allowed'],
        points['points_scored'] < points['points_allowed'],
    ],
    [1.0, 0.0],
    default=0.5,
)

In [108]:
scrimmage_plays['turnover'] = (
    scrimmage_plays['interception'].fillna(0)
    + scrimmage_plays['fumble_lost'].fillna(0)
).clip(upper=1)